# Osteoporosis MobileNetV2 -- Source-Disjoint LOSO CV

## Before running

1. **Upload the 100x100 dataset** as a Kaggle Dataset named `osteo-100x100`:
   - Upload the entire `100x100/` folder (contains `train/`, `valid/`, `test/`).
   - Attach it to this notebook.
   - It will be available at `/kaggle/input/osteo-100x100/100x100`.

2. **Enable GPU** (Accelerator -> GPU T4 x2 or P100) and **enable Internet**.

3. **Confirm `DATASET_ROOT`** in the Config cell below matches your dataset slug.

4. **Smoke test first**: set `SMOKE_TEST = True` in the Config cell, Run All,
   confirm `SMOKE TEST PASSED`, then set `SMOKE_TEST = False`.

5. **Commit for persistence**: use **Save & Run All (Commit)** (top-right menu)
   with background execution. Do NOT rely on an interactive session --
   outputs are lost when the session times out (~9 h on Kaggle). After the
   job completes, download results from the **Output** panel under
   `/kaggle/working/outputs/`.

6. **Resume**: completed folds write `fold_NN_patch_preds.csv` to
   `/kaggle/working/outputs/`. Re-running automatically skips those folds
   and continues from where it stopped.

## What this notebook does

- Clones the repo from GitHub (Internet required) and installs the `osteo` package.
- Builds a fresh manifest from the Kaggle-mounted dataset
  (local `data/manifest.csv` absolute paths are invalid on Kaggle).
- Runs 13-fold Leave-One-Source-Out (LOSO) CV; one MobileNetV2 per fold.
- **Preprocessing**: channel collapse to grayscale + per-image mean-std
  standardisation (removes brightness confound; fixes 1ch/3ch Osteoporosis
  inconsistency).
- **Subsampling**: `TRAIN_SAMPLE_PER_SOURCE` / `EVAL_SAMPLE_PER_SOURCE` patches
  per source -- justified because ~5,775 patches/source are augmentations of a
  single image; subsampling cuts GPU time and equalises per-source counts.
- **Headline metric**: per-image accuracy via majority vote (N=13 sources).
- **Prior 97.28% / F1 figures are INVALID (leaked split)** -- not reproduced.
- Comparisons: (a) 33.3% chance, (b) 53.8% brightness-only LOSO baseline.


In [ ]:
# -- Configuration ----------------------------------------------------------------
# Edit these values before running. All other cells read from these variables.

SMOKE_TEST = False  # True: fold 0, 2 epochs, 50 patches/source; confirm PASSED then False

# Dataset path -- confirm the slug matches your Kaggle Dataset name
DATASET_ROOT = '/kaggle/input/osteo-100x100/100x100'  # <- confirm this slug

# Per-source patch subsampling
# Justified: ~5,775 patches/source are augmentations of ONE source image.
# Subsampling cuts GPU time and equalises per-source representation per fold.
TRAIN_SAMPLE_PER_SOURCE = 800   # max patches per source for train set per fold (None = all)
EVAL_SAMPLE_PER_SOURCE  = 1000  # max patches per source for eval set per fold (None = all)

SEED    = 42
CV_MODE = 'LOSO'
EPOCHS  = 20  # per fold (ignored in smoke test, which uses 2)

OUTPUT_DIR    = '/kaggle/working/outputs'
MANIFEST_PATH = '/kaggle/working/manifest.csv'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Configuration:')
print(f'  SMOKE_TEST              : {SMOKE_TEST}')
print(f'  DATASET_ROOT            : {DATASET_ROOT}')
print(f'  TRAIN_SAMPLE_PER_SOURCE : {TRAIN_SAMPLE_PER_SOURCE}')
print(f'  EVAL_SAMPLE_PER_SOURCE  : {EVAL_SAMPLE_PER_SOURCE}')
print(f'  EPOCHS (this run)       : {2 if SMOKE_TEST else EPOCHS}')
print(f'  OUTPUT_DIR              : {OUTPUT_DIR}')


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Enable GPU Accelerator in Kaggle settings.')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)
if not gpus:
    print('WARNING: TensorFlow sees no GPU -- training will be very slow.')


In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
GIT_URL  = 'https://github.com/MARCUS-00/Early-Detection-of-Osteoporosis-using-Dental-X-rays.git'

if not REPO_DIR.exists():
    print(f'Cloning {GIT_URL} ...')
    subprocess.check_call(['git', 'clone', '--depth', '1', GIT_URL, str(REPO_DIR)])
    print('Clone complete.')
else:
    print('Repo already present (resuming session -- skipping clone).')

# Install the osteo package in editable mode from the cloned repo.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

# Install extras not pre-installed on Kaggle.
# Do NOT pin torch / tensorflow -- they are preinstalled against Kaggle CUDA.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics', 'opencv-python', 'pyyaml',
])
print('All installs complete.')


In [ ]:
import osteo
print('osteo imported from:', osteo.__file__)

from pathlib import Path

missing = []
for split in ['train', 'valid', 'test']:
    for cls in ['Normal', 'Osteopenia', 'Osteoporosis']:
        p = Path(DATASET_ROOT) / split / cls
        n = len(list(p.glob('*.png'))) if p.is_dir() else 0
        status = 'OK' if n > 0 else 'MISSING'
        print(f'  {split}/{cls}: {n:,} files  [{status}]')
        if n == 0:
            missing.append(str(p))

if missing:
    raise FileNotFoundError(
        f'Dataset missing at DATASET_ROOT={DATASET_ROOT!r}. '
        f'Empty/absent dirs: {missing}. '
        f'Check the slug and that the dataset is attached.'
    )
print('Dataset structure OK.')


In [ ]:
from osteo.evaluation.build_manifest import build_manifest

# Build manifest fresh from the Kaggle-mounted dataset.
# Do NOT load the repo's data/manifest.csv -- its paths are local-machine absolute paths.
print(f'Building manifest from {DATASET_ROOT} ...')
manifest = build_manifest(DATASET_ROOT, MANIFEST_PATH)

n_sources = manifest['source_key'].nunique()
assert n_sources == 13, f'Expected 13 sources, got {n_sources} -- check DATASET_ROOT'
print(f'Manifest: {len(manifest):,} rows, {n_sources} sources')
print('Source / class breakdown:')
print(manifest.groupby(['source_key', 'class']).size().to_string())


In [ ]:
_SMOKE_TEST_PASSED = False

if SMOKE_TEST:
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from osteo.evaluation.grouped_cv import make_folds
    from osteo.evaluation.preprocessing import (
        make_preprocess_fn, make_train_datagen, make_eval_datagen, make_flow_kwargs,
    )
    from osteo.classification.model import build_mobilenetv2

    print('=' * 60)
    print('SMOKE TEST (fold 0, 2 epochs, 50 patches/source)')
    print('=' * 60)

    _IMG = (128, 128)
    _BS  = 32

    folds = make_folds(manifest, mode=CV_MODE, seed=SEED)
    train_df, eval_df, _ = folds[0]

    def _tiny_sample(df, n=50, seed=42):
        rng = np.random.default_rng(seed)
        parts = []
        for _, grp in df.groupby('source_key', sort=True):
            idx = rng.choice(len(grp), size=min(n, len(grp)), replace=False)
            parts.append(grp.iloc[sorted(idx)])
        return pd.concat(parts).reset_index(drop=True)

    train_s = _tiny_sample(train_df, 50, SEED)
    eval_s  = _tiny_sample(eval_df,  50, SEED)
    print(f'Smoke patches: train={len(train_s)}, eval={len(eval_s)}')

    preprocess_fn = make_preprocess_fn()
    train_gen_f   = make_train_datagen(preprocess_fn)
    eval_gen_f    = make_eval_datagen(preprocess_fn)

    t_kw = make_flow_kwargs(
        train_s, DATASET_ROOT, train_gen_f,
        shuffle=True, batch_size=_BS, img_size=_IMG, seed=SEED,
    )
    e_kw = make_flow_kwargs(
        eval_s, DATASET_ROOT, eval_gen_f,
        shuffle=False, batch_size=_BS, img_size=_IMG, seed=SEED,
    )
    train_gen = train_gen_f.flow_from_dataframe(**t_kw)
    eval_gen  = eval_gen_f.flow_from_dataframe(**e_kw)

    tf.keras.backend.clear_session()
    model = build_mobilenetv2(num_classes=3, img_size=_IMG)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_gen, epochs=2, validation_data=eval_gen, verbose=1)

    # Verify prediction shape with a synthetic input (avoids generator state issues)
    sample = np.random.randn(1, 128, 128, 3).astype('float32')
    pred   = model.predict(sample, verbose=0)
    assert pred.shape == (1, 3), f'Unexpected prediction shape: {pred.shape}'

    del model
    tf.keras.backend.clear_session()

    _SMOKE_TEST_PASSED = True
    print()
    print('=' * 60)
    print('SMOKE TEST PASSED')
    print('  [OK] manifest built and assertion passed')
    print('  [OK] images found and loaded into generators')
    print('  [OK] two training epochs completed without error')
    print(f'  [OK] prediction shape {pred.shape}')
    print('=' * 60)
    print()
    print('Set SMOKE_TEST = False in the Config cell and re-run for full CV.')
else:
    print('Smoke test skipped (SMOKE_TEST=False). Proceeding to full CV.')


In [ ]:
if not SMOKE_TEST:
    from osteo.evaluation.cv_runner import run_cv

    report_path = run_cv(
        dataset_root            = DATASET_ROOT,
        manifest_path           = MANIFEST_PATH,
        mode                    = CV_MODE,
        epochs                  = EPOCHS,
        output_dir              = OUTPUT_DIR,
        keep_weights            = False,
        train_sample_per_source = TRAIN_SAMPLE_PER_SOURCE,
        eval_sample_per_source  = EVAL_SAMPLE_PER_SOURCE,
        seed                    = SEED,
    )
    print()
    print('Report written to:', report_path)
else:
    print('SMOKE_TEST=True -- full CV skipped. Set SMOKE_TEST=False to train.')


In [ ]:
if not SMOKE_TEST:
    from pathlib import Path
    print(Path(report_path).read_text(encoding='utf-8'))
else:
    print('No report in smoke test mode.')


In [ ]:
from pathlib import Path

output_path = Path(OUTPUT_DIR)
print(f'Files in {OUTPUT_DIR}:')
if output_path.exists():
    files = sorted(output_path.iterdir())
    if files:
        for f in files:
            size_kb = f.stat().st_size // 1024
            print(f'  {f.name:<55}  {size_kb:>6} KB')
    else:
        print('  (empty)')
else:
    print('  (directory does not exist)')

print()
print('To download: Output panel -> outputs/ -> cv_report.txt + cv_report.json')
print('Place both files in outputs/ in your local repo.')


## Saving and downloading results

### Run for persistence

Use **Save & Run All (Commit)** (top-right menu -> Run -> Save & Run All) with
background execution enabled. Do **not** rely on an interactive session: outputs
are lost when the Kaggle session times out (~9 h). After the background job
completes, output files are permanently attached to this notebook version.

### Resuming interrupted runs

If the session times out mid-run, re-running this notebook automatically skips any
fold whose `fold_NN_patch_preds.csv` already exists in `/kaggle/working/outputs/`.
Results from completed folds are loaded and reused; training resumes at the next
incomplete fold.

### Downloading results

1. After the commit job completes, open the **Output** panel.
2. Navigate to `/kaggle/working/outputs/`.
3. Download `cv_report.txt` and `cv_report.json`.
4. Place both files in `outputs/` in your local repo.

### What the report contains (real numbers only -- no fabricated figures)

- Per-source majority-vote predictions (one row per source X-ray, N=13)
- Per-image accuracy vs 33.3% chance and 53.8% brightness-only LOSO baseline
- Per-image confusion matrix
- Brightness-blind Normal test: whether any of the 3 Normal sources were correct
- Per-fold patch accuracy (secondary metric, clearly labelled)
- Limitations: N=13 high variance; proof-of-concept, not a clinical validation
